# Week 2: The Mechanics of Meaning (Tokenization)

This notebook builds intuition for tokenization before implementing tokenizers from scratch.

## Day 1 Goals
1. Compare **word-level**, **character-level**, and **subword** tokenization.
2. Understand why OOV (out-of-vocabulary) happens and how subword methods reduce it.
3. Introduce **byte-level BPE** and why it is robust for emojis, symbols, and multilingual text.
4. Separate **tokenization** from **model input formatting** (special tokens).
5. End with a clear comparison table and practical takeaways for LLMs.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import random
from collections import Counter 

from transformers import AutoTokenizer
from utils import get_imdb_corpus, summarize_one, corpus_token_lengths, word_tokenizer, character_tokenizer, word_piece_tokenizer, bpe_tokenizer
from dotenv import load_dotenv

load_dotenv("../../.env")

True

## 0. Data Setup: IMDB + Stress Test Set

We use IMDB as the main corpus because it is realistic English text with varied vocabulary.

To test robustness, we also add a small stress set with:
- emojis
- accented words
- CJK text
- URLs/hashtags
- punctuation-heavy strings

This combination helps us see where each tokenization strategy breaks or succeeds.


In [2]:
imdb_corpus = get_imdb_corpus()
print(f"Total IMDB reviews: {len(imdb_corpus)}")

random.seed(42)
sample_text = random.choices(imdb_corpus)[0]
print(sample_text[:600])

Total IMDB reviews: 100000
Since most of the political bellyaching about this movie seems to be coming from the self-righteous right, I'm going to ignore it and just talk about the film itself. What starts out as a well-acted exploration of family tensions - especially those between a relatively uneducated set of middle-class parents and their college-educated son - gives way to the clumsy, overwrought melodramatics of the second half of the film. The death of Robert Walker during filming doesn't really excuse or explain the awfulness of the last hour. There's an early scene between John and his mother in which he expla


In [3]:
stress_texts = [
    "I loved it 😂🔥",
    "café naïve résumé",
    "今天天气很好",
    "Check https://example.com #NLP",
    "price=$19.99, don't miss it!",
]

## 1. Word Boundary Tokenization

Word-level tokenization splits text into words (usually by whitespace and punctuation rules).

### Why people use it
- Simple and interpretable
- Fast to prototype

### Main limitation
- Large vocabulary
- Poor handling of rare words, misspellings, and unseen forms (OOV)
- Weak robustness for messy text


In [4]:
word_tokens = word_tokenizer(sample_text)
summarize_one("Word Token", sample_text, word_tokens)
print(f"Unique tokens in sample: {len(set(word_tokens))}")


[Word Token]
text: Since most of the political bellyaching about this movie seems to be coming from the self-...
num_tokens: 282
tokens[:20]: ['Since', 'most', 'of', 'the', 'political', 'bellyaching', 'about', 'this', 'movie', 'seems', 'to', 'be', 'coming', 'from', 'the', 'self-righteous', 'right,', "I'm", 'going', 'to']
Unique tokens in sample: 188


In [5]:
for t in stress_texts:
    summarize_one("Word (stress)", t, word_tokenizer(t), max_show=30)


[Word (stress)]
text: I loved it 😂🔥
num_tokens: 4
tokens[:30]: ['I', 'loved', 'it', '😂🔥']

[Word (stress)]
text: café naïve résumé
num_tokens: 3
tokens[:30]: ['café', 'naïve', 'résumé']

[Word (stress)]
text: 今天天气很好
num_tokens: 1
tokens[:30]: ['今天天气很好']

[Word (stress)]
text: Check https://example.com #NLP
num_tokens: 3
tokens[:30]: ['Check', 'https://example.com', '#NLP']

[Word (stress)]
text: price=$19.99, don't miss it!
num_tokens: 4
tokens[:30]: ['price=$19.99,', "don't", 'miss', 'it!']


## 
2. Character Boundary Tokenization

Character-level tokenization splits text into single characters.

### Strengths
- Very small vocabulary
- Nearly no OOV problem

### Tradeoff
- Sequences become much longer
- Harder for models to capture meaning efficiently
- Higher compute cost for long contexts


In [6]:
char_tokens = character_tokenizer(sample_text)
summarize_one("Character token", sample_text, char_tokens)
print(f"Unique chars in sample: {len(set(char_tokens))}")


[Character token]
text: Since most of the political bellyaching about this movie seems to be coming from the self-...
num_tokens: 1635
tokens[:20]: ['S', 'i', 'n', 'c', 'e', ' ', 'm', 'o', 's', 't', ' ', 'o', 'f', ' ', 't', 'h', 'e', ' ', 'p', 'o']
Unique chars in sample: 53


In [7]:
for t in stress_texts:
    summarize_one("Character (stress)", t, character_tokenizer(t), max_show=40)


[Character (stress)]
text: I loved it 😂🔥
num_tokens: 13
tokens[:40]: ['I', ' ', 'l', 'o', 'v', 'e', 'd', ' ', 'i', 't', ' ', '😂', '🔥']

[Character (stress)]
text: café naïve résumé
num_tokens: 17
tokens[:40]: ['c', 'a', 'f', 'é', ' ', 'n', 'a', 'ï', 'v', 'e', ' ', 'r', 'é', 's', 'u', 'm', 'é']

[Character (stress)]
text: 今天天气很好
num_tokens: 6
tokens[:40]: ['今', '天', '天', '气', '很', '好']

[Character (stress)]
text: Check https://example.com #NLP
num_tokens: 30
tokens[:40]: ['C', 'h', 'e', 'c', 'k', ' ', 'h', 't', 't', 'p', 's', ':', '/', '/', 'e', 'x', 'a', 'm', 'p', 'l', 'e', '.', 'c', 'o', 'm', ' ', '#', 'N', 'L', 'P']

[Character (stress)]
text: price=$19.99, don't miss it!
num_tokens: 28
tokens[:40]: ['p', 'r', 'i', 'c', 'e', '=', '$', '1', '9', '.', '9', '9', ',', ' ', 'd', 'o', 'n', "'", 't', ' ', 'm', 'i', 's', 's', ' ', 'i', 't', '!']


## 3. Subword Tokenization (WordPiece / BPE Family)

Subword tokenization is a practical middle ground:
- Frequent words can stay whole
- Rare words are split into meaningful pieces

Examples:
- `tokenization` -> `token` + `##ization` (WordPiece-style)
- `unaffable` -> smaller parts instead of full OOV

This is why modern NLP and LLM systems rely on subword methods.

### 3A. WordPiece (BERT-style intuition)

WordPiece builds a vocabulary of subword units and applies greedy longest-match segmentation.

### Practical effect
- Reduces OOV compared to word-level
- Keeps sequence lengths shorter than character-level
- Can still emit `[UNK]` for unsupported symbols/scripts depending on vocab


In [8]:
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

bert_tokens = bert_tokenizer.tokenize(sample_text)
summarize_one("WordPiece/Bert", sample_text, bert_tokens)
print(f"Unique tokens in sample:{len(set(bert_tokens))}")


[WordPiece/Bert]
text: Since most of the political bellyaching about this movie seems to be coming from the self-...
num_tokens: 367
tokens[:20]: ['since', 'most', 'of', 'the', 'political', 'belly', '##achi', '##ng', 'about', 'this', 'movie', 'seems', 'to', 'be', 'coming', 'from', 'the', 'self', '-', 'righteous']
Unique tokens in sample:208


In [9]:
for t in stress_texts:
    toks = bert_tokenizer.tokenize(t)
    summarize_one("WordPiece/BERT (stress)", t, toks, max_show=30)
    has_unk = "[UNK]" in toks
    print(f"has_[UNK]={has_unk:5} | text={t}")


[WordPiece/BERT (stress)]
text: I loved it 😂🔥
num_tokens: 4
tokens[:30]: ['i', 'loved', 'it', '[UNK]']
has_[UNK]=    1 | text=I loved it 😂🔥

[WordPiece/BERT (stress)]
text: café naïve résumé
num_tokens: 3
tokens[:30]: ['cafe', 'naive', 'resume']
has_[UNK]=    0 | text=café naïve résumé

[WordPiece/BERT (stress)]
text: 今天天气很好
num_tokens: 6
tokens[:30]: ['[UNK]', '天', '天', '[UNK]', '[UNK]', '[UNK]']
has_[UNK]=    1 | text=今天天气很好

[WordPiece/BERT (stress)]
text: Check https://example.com #NLP
num_tokens: 11
tokens[:30]: ['check', 'https', ':', '/', '/', 'example', '.', 'com', '#', 'nl', '##p']
has_[UNK]=    0 | text=Check https://example.com #NLP

[WordPiece/BERT (stress)]
text: price=$19.99, don't miss it!
num_tokens: 13
tokens[:30]: ['price', '=', '$', '19', '.', '99', ',', 'don', "'", 't', 'miss', 'it', '!']
has_[UNK]=    0 | text=price=$19.99, don't miss it!


### 3B. Byte-Level BPE (GPT-style intuition)

Byte-level BPE starts from UTF-8 bytes, then learns merges.

### Why this matters
- Any text can be represented as bytes
- Extremely robust for emojis, symbols, multilingual text, and noisy inputs
- Avoids hard failures from unknown characters


In [10]:
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")

gpt2_tokens = gpt2_tokenizer.tokenize(sample_text)
summarize_one("Byte-Level BPE/GPT-2", sample_text, gpt2_tokens)
print(f"Unique tokens in sample:{len(set(gpt2_tokens))}")


[Byte-Level BPE/GPT-2]
text: Since most of the political bellyaching about this movie seems to be coming from the self-...
num_tokens: 350
tokens[:20]: ['Since', 'Ġmost', 'Ġof', 'Ġthe', 'Ġpolitical', 'Ġbelly', 'aching', 'Ġabout', 'Ġthis', 'Ġmovie', 'Ġseems', 'Ġto', 'Ġbe', 'Ġcoming', 'Ġfrom', 'Ġthe', 'Ġself', '-', 'righteous', 'Ġright']
Unique tokens in sample:217


In [11]:
for t in stress_texts:
    toks = gpt2_tokenizer.tokenize(t)
    summarize_one("Byte-Level BPE/GPT-2 (stress)", t, toks, max_show=30)



[Byte-Level BPE/GPT-2 (stress)]
text: I loved it 😂🔥
num_tokens: 8
tokens[:30]: ['I', 'Ġloved', 'Ġit', 'ĠðŁĺ', 'Ĥ', 'ðŁ', 'Ķ', '¥']

[Byte-Level BPE/GPT-2 (stress)]
text: café naïve résumé
num_tokens: 7
tokens[:30]: ['c', 'af', 'Ã©', 'ĠnaÃ¯ve', 'ĠrÃ©', 'sum', 'Ã©']

[Byte-Level BPE/GPT-2 (stress)]
text: 今天天气很好
num_tokens: 10
tokens[:30]: ['ä»', 'Ĭ', 'å¤©', 'å¤©', 'æ°', 'Ķ', 'å¾', 'Ī', 'å¥', '½']

[Byte-Level BPE/GPT-2 (stress)]
text: Check https://example.com #NLP
num_tokens: 9
tokens[:30]: ['Check', 'Ġhttps', '://', 'example', '.', 'com', 'Ġ#', 'N', 'LP']

[Byte-Level BPE/GPT-2 (stress)]
text: price=$19.99, don't miss it!
num_tokens: 11
tokens[:30]: ['price', '=$', '19', '.', '99', ',', 'Ġdon', "'t", 'Ġmiss', 'Ġit', '!']


## 4. Special Tokens: Packaging for Model Inputs (Not Raw Corpus)

Important distinction:

- **Tokenization**: split text into token units
- **Sequence packaging**: add model control tokens such as `[CLS]`, `[SEP]`, `<bos>`, `<eos>`, `<pad>`

We add special tokens when preparing model inputs, not when building raw corpus text.


In [12]:
text_sample = "Hello world!"

bert_ids_plain = bert_tokenizer.encode(text_sample, add_special_tokens=False)
bert_ids_special = bert_tokenizer.encode(text_sample, add_special_tokens=True)

print("BERT plain tokens:   ", bert_tokenizer.convert_ids_to_tokens(bert_ids_plain))
print("BERT special tokens: ", bert_tokenizer.convert_ids_to_tokens(bert_ids_special))


BERT plain tokens:    ['hello', 'world', '!']
BERT special tokens:  ['[CLS]', 'hello', 'world', '!', '[SEP]']


In [13]:
# GPT-2 often has no default BOS/EOS in simple encode call
gpt2_ids = gpt2_tokenizer.encode(text_sample, add_special_tokens=True)
print("GPT-2 tokens:", gpt2_tokenizer.convert_ids_to_tokens(gpt2_ids))
print("bos_token:", gpt2_tokenizer.bos_token, "eos_token:", gpt2_tokenizer.eos_token, "pad_token:", gpt2_tokenizer.pad_token)


GPT-2 tokens: ['Hello', 'Ġworld', '!']
bos_token: <|endoftext|> eos_token: <|endoftext|> pad_token: None


## 5. Quick Comparison on the Same Inputs

For each method, compare:
- Number of tokens (sequence length)
- Whether unknown tokens appear
- Behavior on stress-test texts
- Decoding quality (encode -> decode)

This turns tokenization discussion from theory into measurable behavior.


In [14]:
# Use a small subset for speed
subset = imdb_corpus[:2000]

metrics = {
    "word": corpus_token_lengths(subset, word_tokenizer),
    "char": corpus_token_lengths(subset, character_tokenizer),
    "bert_wordpiece": corpus_token_lengths(subset, lambda t: bert_tokenizer.tokenize(t)),
    "gpt2_byte_bpe": corpus_token_lengths(subset, lambda t: gpt2_tokenizer.tokenize(t)),
}

metrics


Token indices sequence length is longer than the specified maximum sequence length for this model (718 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1168 > 1024). Running this sequence through the model will result in indexing errors


{'word': {'avg_len': 225.78,
  'median_len': 171.0,
  'min_len': 11,
  'max_len': 1263},
 'char': {'avg_len': 1276.16,
  'median_len': 964.0,
  'min_len': 65,
  'max_len': 7382},
 'bert_wordpiece': {'avg_len': 303.84,
  'median_len': 230.0,
  'min_len': 19,
  'max_len': 1681},
 'gpt2_byte_bpe': {'avg_len': 290.47,
  'median_len': 221.0,
  'min_len': 16,
  'max_len': 1659}}

In [15]:
print("\n--- Single-text Summary ---")
print(f"Original words:      {len(sample_text.split())}")
print(f"Original characters: {len(sample_text)}")
print(f"Word tokens:         {len(word_tokens)}")
print(f"Character tokens:    {len(char_tokens)}")
print(f"BERT tokens:         {len(bert_tokens)}")
print(f"GPT-2 tokens:        {len(gpt2_tokens)}")



--- Single-text Summary ---
Original words:      282
Original characters: 1635
Word tokens:         282
Character tokens:    1635
BERT tokens:         367
GPT-2 tokens:        350


In [16]:
print("\n--- Corpus Length Stats (first 2,000 IMDB texts) ---")
for name, m in metrics.items():
    print(f"{name:16} avg={m['avg_len']:<8} median={m['median_len']:<8} min={m['min_len']:<6} max={m['max_len']}")



--- Corpus Length Stats (first 2,000 IMDB texts) ---
word             avg=225.78   median=171.0    min=11     max=1263
char             avg=1276.16  median=964.0    min=65     max=7382
bert_wordpiece   avg=303.84   median=230.0    min=19     max=1681
gpt2_byte_bpe    avg=290.47   median=221.0    min=16     max=1659


In [17]:
comparison_rows = [
    ("Word", "High", "Largest", "Shortest", "Weak", "Weak"),
    ("Character", "Very Low", "Smallest", "Longest", "Strong", "Strong"),
    ("WordPiece", "Lower than word", "Medium", "Medium", "Medium", "Medium"),
    ("Byte-level BPE", "Very Low", "Medium", "Medium", "Strong", "Strong"),
]

for r in comparison_rows:
    print(f"{r[0]:15} | OOV risk={r[1]:16} | Vocab={r[2]:8} | SeqLen={r[3]:8} | Multilingual={r[4]:6} | Emoji/Symbols={r[5]}")


Word            | OOV risk=High             | Vocab=Largest  | SeqLen=Shortest | Multilingual=Weak   | Emoji/Symbols=Weak
Character       | OOV risk=Very Low         | Vocab=Smallest | SeqLen=Longest  | Multilingual=Strong | Emoji/Symbols=Strong
WordPiece       | OOV risk=Lower than word  | Vocab=Medium   | SeqLen=Medium   | Multilingual=Medium | Emoji/Symbols=Medium
Byte-level BPE  | OOV risk=Very Low         | Vocab=Medium   | SeqLen=Medium   | Multilingual=Strong | Emoji/Symbols=Strong


## 6. Summary: What We Learned Today

- Word-level is simple but brittle (OOV-heavy).
- Character-level is robust but inefficient (long sequences).
- Subword methods are the practical compromise.
- Byte-level BPE is especially robust for real-world messy text.
- Special tokens belong to model input formatting, not corpus construction.

This prepares us for Day 2: implementing **BPE** and **WordPiece** from scratch.
